In [1]:
import duckdb

print("DuckDB:", duckdb.__version__)
print("Jupyter работает")


DuckDB: 1.2.0
Jupyter работает


# Лабораторная работа № 3

## Профилирование и контроль качества данных

**Набор данных:** Seoul Bike Sharing Demand, UCI.

**Единица наблюдения:** один час работы системы велопроката Seoul Bike.

**Бизнес-ключ:** сочетание полей `Date` и `Hour`.

**Аналитическая задача:** определить, как почасовое количество арендованных велосипедов различается в зависимости от времени, календарных и погодных условий.

В лабораторной работе выполняется профилирование исходного слоя `raw`, проверка качества данных и формирование слоёв `staging` и `quarantine`.


## 1. Параметры и подключение к исходным данным

Исходные данные читаются непосредственно из слоя `raw` объектного хранилища SeaweedFS.

Входной объект:

`s3://raw/seoul_bike_sharing_demand/ingested_on=2026-10-13/SeoulBikeData.csv`

Результаты лабораторной работы будут сохранены в слой `staging` с сохранением даты загрузки исходного raw-объекта.


In [2]:
RAW_PATH = (
    "s3://raw/seoul_bike_sharing_demand/"
    "ingested_on=2026-10-13/"
    "SeoulBikeData.csv"
)

SOURCE_INGESTED_ON = "2026-10-13"

STAGING_PATH = (
    "s3://staging/seoul_bike_sharing_demand/"
    f"source_ingested_on={SOURCE_INGESTED_ON}/"
    "staging.parquet"
)

QUARANTINE_PATH = (
    "s3://staging/seoul_bike_sharing_demand/"
    f"source_ingested_on={SOURCE_INGESTED_ON}/"
    "quarantine.parquet"
)

print("RAW:", RAW_PATH)
print("STAGING:", STAGING_PATH)
print("QUARANTINE:", QUARANTINE_PATH)


RAW: s3://raw/seoul_bike_sharing_demand/ingested_on=2026-10-13/SeoulBikeData.csv
STAGING: s3://staging/seoul_bike_sharing_demand/source_ingested_on=2026-10-13/staging.parquet
QUARANTINE: s3://staging/seoul_bike_sharing_demand/source_ingested_on=2026-10-13/quarantine.parquet


In [3]:
import os

from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(".env", usecwd=True)

if not env_path:
    raise FileNotFoundError("Файл .env не найден")

load_dotenv(env_path)

print("Используется .env:", env_path)
print("S3 endpoint:", os.getenv("S3_ENDPOINT"))


Используется .env: c:\Users\Андрей\Desktop\itad_lab2\itad-labs\.env
S3 endpoint: localhost:8333


In [4]:
import duckdb

connection = duckdb.connect()

connection.execute("INSTALL httpfs")
connection.execute("LOAD httpfs")

s3_endpoint = os.environ["S3_ENDPOINT"]
s3_access_key = os.environ["S3_ACCESS_KEY"]
s3_secret_key = os.environ["S3_SECRET_KEY"]

connection.execute(f"SET s3_endpoint='{s3_endpoint}'")
connection.execute("SET s3_url_style='path'")
connection.execute("SET s3_use_ssl=false")
connection.execute("SET s3_region='us-east-1'")
connection.execute(f"SET s3_access_key_id='{s3_access_key}'")
connection.execute(f"SET s3_secret_access_key='{s3_secret_key}'")

print("Подключение DuckDB к SeaweedFS настроено")


Подключение DuckDB к SeaweedFS настроено


In [5]:
connection.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW raw_data AS
    SELECT *
    FROM read_csv_auto(
        '{RAW_PATH}',
        encoding='latin-1',
        hive_partitioning=true
    )
    """
)

print("Представление raw_data создано")


Представление raw_data создано


In [6]:
connection.sql(
    """
    SELECT *
    FROM raw_data
    LIMIT 5
    """
).show()

┌────────────┬───────────────────┬───────┬─────────────────┬─────────────┬──────────────────┬──────────────────┬───────────────────────────┬─────────────────────────┬──────────────┬───────────────┬─────────┬────────────┬─────────────────┬─────────────┐
│    Date    │ Rented Bike Count │ Hour  │ Temperature(°C) │ Humidity(%) │ Wind speed (m/s) │ Visibility (10m) │ Dew point temperature(°C) │ Solar Radiation (MJ/m2) │ Rainfall(mm) │ Snowfall (cm) │ Seasons │  Holiday   │ Functioning Day │ ingested_on │
│    date    │       int64       │ int64 │     double      │    int64    │      double      │      int64       │          double           │         double          │    double    │    double     │ varchar │  varchar   │     boolean     │    date     │
├────────────┼───────────────────┼───────┼─────────────────┼─────────────┼──────────────────┼──────────────────┼───────────────────────────┼─────────────────────────┼──────────────┼───────────────┼─────────┼────────────┼─────────────────┼───

## 2. Профилирование исходных данных

На данном этапе выполняется первичный анализ структуры и качества исходного набора данных до применения правил очистки.

Проверяются:
- количество строк и столбцов;
- типы данных;
- наличие пропусков;
- дубликаты по бизнес-ключу `Date + Hour`;
- временной диапазон;
- распределение наблюдений по часам суток.


In [7]:
connection.sql(
    """
    SELECT COUNT(*) AS row_count
    FROM raw_data
    """
).show()


┌───────────┐
│ row_count │
│   int64   │
├───────────┤
│      8760 │
└───────────┘



In [8]:
connection.sql(
    """
    DESCRIBE raw_data
    """
).show()


┌───────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name        │ column_type │  null   │   key   │ default │  extra  │
│          varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Date                      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ Rented Bike Count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Hour                      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Temperature(°C)           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Humidity(%)               │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Wind speed (m/s)          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Visibility (10m)          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Dew point temperature(°C) │ DOUBLE      │ YES     │ NULL    │ NULL    │ NU

In [9]:
connection.sql(
    """
    SELECT
        MIN(Date) AS min_date,
        MAX(Date) AS max_date
    FROM raw_data
    """
).show()


┌────────────┬────────────┐
│  min_date  │  max_date  │
│    date    │    date    │
├────────────┼────────────┤
│ 2017-12-01 │ 2018-11-30 │
└────────────┴────────────┘



In [10]:
connection.sql(
    """
    SELECT
        COUNT(*) AS duplicate_groups
    FROM (
        SELECT
            Date,
            Hour,
            COUNT(*) AS cnt
        FROM raw_data
        GROUP BY Date, Hour
        HAVING COUNT(*) > 1
    )
    """
).show()


┌──────────────────┐
│ duplicate_groups │
│      int64       │
├──────────────────┤
│                0 │
└──────────────────┘



In [11]:
connection.sql(
    """
    SELECT
        COUNT(*) AS total_rows,

        SUM(CASE WHEN Date IS NULL THEN 1 ELSE 0 END)
            AS date_nulls,

        SUM(CASE WHEN Hour IS NULL THEN 1 ELSE 0 END)
            AS hour_nulls,

        SUM(
            CASE
                WHEN "Rented Bike Count" IS NULL THEN 1
                ELSE 0
            END
        ) AS rented_bike_count_nulls,

        SUM(
            CASE
                WHEN "Humidity(%)" IS NULL THEN 1
                ELSE 0
            END
        ) AS humidity_nulls,

        SUM(
            CASE
                WHEN "Functioning Day" IS NULL THEN 1
                ELSE 0
            END
        ) AS functioning_day_nulls

    FROM raw_data
    """
).show()


┌────────────┬────────────┬────────────┬─────────────────────────┬────────────────┬───────────────────────┐
│ total_rows │ date_nulls │ hour_nulls │ rented_bike_count_nulls │ humidity_nulls │ functioning_day_nulls │
│   int64    │   int128   │   int128   │         int128          │     int128     │        int128         │
├────────────┼────────────┼────────────┼─────────────────────────┼────────────────┼───────────────────────┤
│       8760 │          0 │          0 │                       0 │              0 │                     0 │
└────────────┴────────────┴────────────┴─────────────────────────┴────────────────┴───────────────────────┘



In [12]:
connection.sql(
    """
    SELECT
        MIN(Hour) AS min_hour,
        MAX(Hour) AS max_hour,

        MIN("Rented Bike Count") AS min_rented,
        MAX("Rented Bike Count") AS max_rented,

        MIN("Humidity(%)") AS min_humidity,
        MAX("Humidity(%)") AS max_humidity

    FROM raw_data
    """
).show()


┌──────────┬──────────┬────────────┬────────────┬──────────────┬──────────────┐
│ min_hour │ max_hour │ min_rented │ max_rented │ min_humidity │ max_humidity │
│  int64   │  int64   │   int64    │   int64    │    int64     │    int64     │
├──────────┼──────────┼────────────┼────────────┼──────────────┼──────────────┤
│        0 │       23 │          0 │       3556 │            0 │           98 │
└──────────┴──────────┴────────────┴────────────┴──────────────┴──────────────┘



In [13]:
connection.sql(
    """
    SELECT
        Hour,
        COUNT(*) AS observations
    FROM raw_data
    GROUP BY Hour
    ORDER BY Hour
    """
).show()


┌───────┬──────────────┐
│ Hour  │ observations │
│ int64 │    int64     │
├───────┼──────────────┤
│     0 │          365 │
│     1 │          365 │
│     2 │          365 │
│     3 │          365 │
│     4 │          365 │
│     5 │          365 │
│     6 │          365 │
│     7 │          365 │
│     8 │          365 │
│     9 │          365 │
│     · │           ·  │
│     · │           ·  │
│     · │           ·  │
│    14 │          365 │
│    15 │          365 │
│    16 │          365 │
│    17 │          365 │
│    18 │          365 │
│    19 │          365 │
│    20 │          365 │
│    21 │          365 │
│    22 │          365 │
│    23 │          365 │
├───────┴──────────────┤
│  24 rows (20 shown)  │
└──────────────────────┘



### Предварительные выводы профилирования

По результатам первичного профилирования необходимо оценить:
- полноту набора данных;
- наличие дубликатов по ключу `Date + Hour`;
- наличие пропусков;
- корректность диапазонов основных числовых полей;
- полноту временного покрытия.

Конкретные выводы будут сформулированы после выполнения запросов выше.


### Вывод по профилированию

Исходный набор содержит 8760 почасовых наблюдений.

Дубликаты по бизнес-ключу `Date + Hour` отсутствуют. Пропуски в полях `Date`, `Hour`, `Rented Bike Count`, `Humidity(%)` и `Functioning Day` не обнаружены.

Поле `Hour` находится в допустимом диапазоне от 0 до 23. Количество арендованных велосипедов принимает значения от 0 до 3556, отрицательные значения отсутствуют. Влажность находится в диапазоне от 0 до 98 %.

Каждый из 24 часов суток представлен 365 наблюдениями, что соответствует полному годовому циклу данных.

По результатам первичного профилирования явных структурных проблем в выбранных полях не обнаружено.


## 3. Правила качества данных

Для контроля качества данных используются правила, связанные с аналитической задачей и структурой набора.

Для каждого правила задаются:
- идентификатор;
- проверяемое условие;
- допустимый порог нарушений;
- критичность;
- действие при нарушении.

Критические нарушения с действием `quarantine` исключаются из слоя `staging` и сохраняются в `quarantine`.

Предупреждения с действием `flag` не приводят к исключению строки, но отмечаются дополнительным признаком качества.


### Правило `unique_date_hour`

**Условие:** сочетание `Date + Hour` должно быть уникальным.

**Порог:** 0 нарушений.

**Критичность:** critical.

**Действие:** quarantine.

Правило необходимо, поскольку одна строка набора должна соответствовать одному уникальному часу наблюдения.


In [14]:
UNIQUE_DATE_HOUR_THRESHOLD = 0

connection.sql(
    f"""
    SELECT
        COUNT(*) AS checked_rows,
        COUNT(*) - COUNT(DISTINCT (Date, Hour)) AS violation_count,
        {UNIQUE_DATE_HOUR_THRESHOLD} AS threshold,
        (
            COUNT(*) - COUNT(DISTINCT (Date, Hour))
            > {UNIQUE_DATE_HOUR_THRESHOLD}
        ) AS threshold_exceeded
    FROM raw_data
    """
).show()


┌──────────────┬─────────────────┬───────────┬────────────────────┐
│ checked_rows │ violation_count │ threshold │ threshold_exceeded │
│    int64     │      int64      │   int32   │      boolean       │
├──────────────┼─────────────────┼───────────┼────────────────────┤
│         8760 │               0 │         0 │ false              │
└──────────────┴─────────────────┴───────────┴────────────────────┘



### Правило `rented_count_not_null`

**Условие:** поле `Rented Bike Count` обязательно должно быть заполнено.

**Порог:** 0 нарушений.

**Критичность:** critical.

**Действие:** quarantine.

Без значения количества аренд строка не может использоваться для анализа спроса.


In [15]:
RENTED_COUNT_NULL_THRESHOLD = 0

connection.sql(
    f"""
    SELECT
        COUNT(*) AS checked_rows,
        SUM(
            CASE
                WHEN "Rented Bike Count" IS NULL THEN 1
                ELSE 0
            END
        ) AS violation_count,
        {RENTED_COUNT_NULL_THRESHOLD} AS threshold,
        (
            SUM(
                CASE
                    WHEN "Rented Bike Count" IS NULL THEN 1
                    ELSE 0
                END
            ) > {RENTED_COUNT_NULL_THRESHOLD}
        ) AS threshold_exceeded
    FROM raw_data
    """
).show()


┌──────────────┬─────────────────┬───────────┬────────────────────┐
│ checked_rows │ violation_count │ threshold │ threshold_exceeded │
│    int64     │     int128      │   int32   │      boolean       │
├──────────────┼─────────────────┼───────────┼────────────────────┤
│         8760 │               0 │         0 │ false              │
└──────────────┴─────────────────┴───────────┴────────────────────┘



### Правило `valid_hour`

**Условие:** значение `Hour` должно находиться в диапазоне от 0 до 23.

**Порог:** 0 нарушений.

**Критичность:** critical.

**Действие:** quarantine.


In [16]:
VALID_HOUR_THRESHOLD = 0

connection.sql(
    f"""
    SELECT
        COUNT(*) AS checked_rows,
        SUM(
            CASE
                WHEN Hour IS NULL OR Hour < 0 OR Hour > 23 THEN 1
                ELSE 0
            END
        ) AS violation_count,
        {VALID_HOUR_THRESHOLD} AS threshold,
        (
            SUM(
                CASE
                    WHEN Hour IS NULL OR Hour < 0 OR Hour > 23 THEN 1
                    ELSE 0
                END
            ) > {VALID_HOUR_THRESHOLD}
        ) AS threshold_exceeded
    FROM raw_data
    """
).show()


┌──────────────┬─────────────────┬───────────┬────────────────────┐
│ checked_rows │ violation_count │ threshold │ threshold_exceeded │
│    int64     │     int128      │   int32   │      boolean       │
├──────────────┼─────────────────┼───────────┼────────────────────┤
│         8760 │               0 │         0 │ false              │
└──────────────┴─────────────────┴───────────┴────────────────────┘



### Правило `non_negative_rented_count`

**Условие:** `Rented Bike Count` должно быть больше либо равно 0.

**Порог:** 0 нарушений.

**Критичность:** critical.

**Действие:** quarantine.

Отрицательное количество аренд не имеет физического смысла.


In [17]:
NON_NEGATIVE_RENTED_THRESHOLD = 0

connection.sql(
    f"""
    SELECT
        COUNT(*) AS checked_rows,
        SUM(
            CASE
                WHEN "Rented Bike Count" < 0 THEN 1
                ELSE 0
            END
        ) AS violation_count,
        {NON_NEGATIVE_RENTED_THRESHOLD} AS threshold,
        (
            SUM(
                CASE
                    WHEN "Rented Bike Count" < 0 THEN 1
                    ELSE 0
                END
            ) > {NON_NEGATIVE_RENTED_THRESHOLD}
        ) AS threshold_exceeded
    FROM raw_data
    """
).show()


┌──────────────┬─────────────────┬───────────┬────────────────────┐
│ checked_rows │ violation_count │ threshold │ threshold_exceeded │
│    int64     │     int128      │   int32   │      boolean       │
├──────────────┼─────────────────┼───────────┼────────────────────┤
│         8760 │               0 │         0 │ false              │
└──────────────┴─────────────────┴───────────┴────────────────────┘



### Правило `valid_humidity`

**Условие:** относительная влажность должна находиться в диапазоне от 0 до 100 %.

**Порог:** 0 нарушений.

**Критичность:** warning.

**Действие:** flag.

Значения вне физически допустимого диапазона требуют дополнительной проверки, но строка сохраняется для анализа.


In [18]:
VALID_HUMIDITY_THRESHOLD = 0

connection.sql(
    f"""
    SELECT
        COUNT(*) AS checked_rows,
        SUM(
            CASE
                WHEN "Humidity(%)" < 0
                     OR "Humidity(%)" > 100
                THEN 1
                ELSE 0
            END
        ) AS violation_count,
        {VALID_HUMIDITY_THRESHOLD} AS threshold,
        (
            SUM(
                CASE
                    WHEN "Humidity(%)" < 0
                         OR "Humidity(%)" > 100
                    THEN 1
                    ELSE 0
                END
            ) > {VALID_HUMIDITY_THRESHOLD}
        ) AS threshold_exceeded
    FROM raw_data
    """
).show()


┌──────────────┬─────────────────┬───────────┬────────────────────┐
│ checked_rows │ violation_count │ threshold │ threshold_exceeded │
│    int64     │     int128      │   int32   │      boolean       │
├──────────────┼─────────────────┼───────────┼────────────────────┤
│         8760 │               0 │         0 │ false              │
└──────────────┴─────────────────┴───────────┴────────────────────┘



In [19]:
connection.sql(
    """
    SELECT
        "Functioning Day",
        COUNT(*) AS rows_count,
        MIN("Rented Bike Count") AS min_rented,
        MAX("Rented Bike Count") AS max_rented,
        SUM("Rented Bike Count") AS total_rented
    FROM raw_data
    GROUP BY "Functioning Day"
    ORDER BY "Functioning Day"
    """
).show()


┌─────────────────┬────────────┬────────────┬────────────┬──────────────┐
│ Functioning Day │ rows_count │ min_rented │ max_rented │ total_rented │
│     boolean     │   int64    │   int64    │   int64    │    int128    │
├─────────────────┼────────────┼────────────┼────────────┼──────────────┤
│ false           │        295 │          0 │          0 │            0 │
│ true            │       8465 │          2 │       3556 │      6172314 │
└─────────────────┴────────────┴────────────┴────────────┴──────────────┘



### Правило `functioning_day_consistency`

**Условие:** если `Functioning Day = false`, то `Rented Bike Count` должно быть равно 0.

**Порог:** 0 нарушений.

**Критичность:** warning.

**Действие:** flag.

Правило проверяет согласованность признака функционирования сервиса и количества зарегистрированных аренд. Если сервис не функционирует, количество аренд должно быть равно нулю.


In [20]:
FUNCTIONING_DAY_THRESHOLD = 0

connection.sql(
    f"""
    SELECT
        COUNT(*) FILTER (
            WHERE "Functioning Day" = false
        ) AS checked_rows,

        COUNT(*) FILTER (
            WHERE "Functioning Day" = false
              AND "Rented Bike Count" <> 0
        ) AS violation_count,

        {FUNCTIONING_DAY_THRESHOLD} AS threshold,

        (
            COUNT(*) FILTER (
                WHERE "Functioning Day" = false
                  AND "Rented Bike Count" <> 0
            ) > {FUNCTIONING_DAY_THRESHOLD}
        ) AS threshold_exceeded

    FROM raw_data
    """
).show()


┌──────────────┬─────────────────┬───────────┬────────────────────┐
│ checked_rows │ violation_count │ threshold │ threshold_exceeded │
│    int64     │      int64      │   int32   │      boolean       │
├──────────────┼─────────────────┼───────────┼────────────────────┤
│          295 │               0 │         0 │ false              │
└──────────────┴─────────────────┴───────────┴────────────────────┘



### Сводные результаты проверок качества

В таблице ниже представлены результаты всех реализованных правил качества данных.

Для каждого правила указаны:
- количество проверенных строк;
- количество нарушений;
- доля нарушений;
- допустимый порог;
- факт превышения порога;
- критичность;
- действие при нарушении.


In [21]:
connection.sql(
    f"""
    SELECT
        'unique_date_hour' AS rule_id,
        COUNT(*) AS checked_rows,
        COUNT(*) - COUNT(DISTINCT (Date, Hour)) AS violation_count,
        (
            COUNT(*) - COUNT(DISTINCT (Date, Hour))
        )::DOUBLE / COUNT(*) AS violation_rate,
        {UNIQUE_DATE_HOUR_THRESHOLD} AS threshold,
        (
            COUNT(*) - COUNT(DISTINCT (Date, Hour))
            > {UNIQUE_DATE_HOUR_THRESHOLD}
        ) AS threshold_exceeded,
        'critical' AS severity,
        'quarantine' AS action
    FROM raw_data

    UNION ALL

    SELECT
        'rented_count_not_null',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE "Rented Bike Count" IS NULL
        ),
        (
            COUNT(*) FILTER (
                WHERE "Rented Bike Count" IS NULL
            )
        )::DOUBLE / COUNT(*),
        {RENTED_COUNT_NULL_THRESHOLD},
        (
            COUNT(*) FILTER (
                WHERE "Rented Bike Count" IS NULL
            ) > {RENTED_COUNT_NULL_THRESHOLD}
        ),
        'critical',
        'quarantine'
    FROM raw_data

    UNION ALL

    SELECT
        'valid_hour',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE Hour IS NULL
               OR Hour < 0
               OR Hour > 23
        ),
        (
            COUNT(*) FILTER (
                WHERE Hour IS NULL
                   OR Hour < 0
                   OR Hour > 23
            )
        )::DOUBLE / COUNT(*),
        {VALID_HOUR_THRESHOLD},
        (
            COUNT(*) FILTER (
                WHERE Hour IS NULL
                   OR Hour < 0
                   OR Hour > 23
            ) > {VALID_HOUR_THRESHOLD}
        ),
        'critical',
        'quarantine'
    FROM raw_data

    UNION ALL

    SELECT
        'non_negative_rented_count',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE "Rented Bike Count" < 0
        ),
        (
            COUNT(*) FILTER (
                WHERE "Rented Bike Count" < 0
            )
        )::DOUBLE / COUNT(*),
        {NON_NEGATIVE_RENTED_THRESHOLD},
        (
            COUNT(*) FILTER (
                WHERE "Rented Bike Count" < 0
            ) > {NON_NEGATIVE_RENTED_THRESHOLD}
        ),
        'critical',
        'quarantine'
    FROM raw_data

    UNION ALL

    SELECT
        'valid_humidity',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE "Humidity(%)" < 0
               OR "Humidity(%)" > 100
        ),
        (
            COUNT(*) FILTER (
                WHERE "Humidity(%)" < 0
                   OR "Humidity(%)" > 100
            )
        )::DOUBLE / COUNT(*),
        {VALID_HUMIDITY_THRESHOLD},
        (
            COUNT(*) FILTER (
                WHERE "Humidity(%)" < 0
                   OR "Humidity(%)" > 100
            ) > {VALID_HUMIDITY_THRESHOLD}
        ),
        'warning',
        'flag'
    FROM raw_data

    UNION ALL

    SELECT
        'functioning_day_consistency',
        COUNT(*) FILTER (
            WHERE "Functioning Day" = false
        ),
        COUNT(*) FILTER (
            WHERE "Functioning Day" = false
              AND "Rented Bike Count" <> 0
        ),
        CASE
            WHEN COUNT(*) FILTER (
                WHERE "Functioning Day" = false
            ) = 0
            THEN 0
            ELSE (
                COUNT(*) FILTER (
                    WHERE "Functioning Day" = false
                      AND "Rented Bike Count" <> 0
                )
            )::DOUBLE
            /
            COUNT(*) FILTER (
                WHERE "Functioning Day" = false
            )
        END,
        {FUNCTIONING_DAY_THRESHOLD},
        (
            COUNT(*) FILTER (
                WHERE "Functioning Day" = false
                  AND "Rented Bike Count" <> 0
            ) > {FUNCTIONING_DAY_THRESHOLD}
        ),
        'warning',
        'flag'
    FROM raw_data
    """
).show()


┌─────────────────────────────┬──────────────┬─────────────────┬────────────────┬───────────┬────────────────────┬──────────┬────────────┐
│           rule_id           │ checked_rows │ violation_count │ violation_rate │ threshold │ threshold_exceeded │ severity │   action   │
│           varchar           │    int64     │      int64      │     double     │   int32   │      boolean       │ varchar  │  varchar   │
├─────────────────────────────┼──────────────┼─────────────────┼────────────────┼───────────┼────────────────────┼──────────┼────────────┤
│ unique_date_hour            │         8760 │               0 │            0.0 │         0 │ false              │ critical │ quarantine │
│ rented_count_not_null       │         8760 │               0 │            0.0 │         0 │ false              │ critical │ quarantine │
│ valid_hour                  │         8760 │               0 │            0.0 │         0 │ false              │ critical │ quarantine │
│ non_negative_rented_count

### Вывод по контролю качества

Все реализованные правила качества успешно выполнены.

Критические нарушения, требующие помещения строк в карантин, не обнаружены. Также не обнаружены нарушения предупреждающих правил.

Набор данных может быть использован для формирования нормализованного слоя `staging`.


## 4. Формирование staging и quarantine

На основании результатов проверок качества каждой строке назначаются признаки нарушений.

Строки с критическими нарушениями исключаются из `staging` и помещаются в `quarantine`.

Предупреждающие нарушения не исключают строку, а сохраняются в `staging` в виде специальных quality-флагов.


In [22]:
connection.execute(
    """
    CREATE OR REPLACE TEMP VIEW quality_evaluated AS

    WITH base AS (
        SELECT
            *,
            COUNT(*) OVER (
                PARTITION BY Date, Hour
            ) AS date_hour_count
        FROM raw_data
    )

    SELECT
        *,

        date_hour_count > 1
            AS q_unique_date_hour,

        "Rented Bike Count" IS NULL
            AS q_rented_count_not_null,

        (
            Hour IS NULL
            OR Hour < 0
            OR Hour > 23
        ) AS q_valid_hour,

        COALESCE(
            "Rented Bike Count" < 0,
            false
        ) AS q_non_negative_rented_count,

        COALESCE(
            "Humidity(%)" < 0
            OR "Humidity(%)" > 100,
            false
        ) AS q_valid_humidity,

        COALESCE(
            "Functioning Day" = false
            AND "Rented Bike Count" <> 0,
            false
        ) AS q_functioning_day_consistency,

        (
            date_hour_count > 1

            OR "Rented Bike Count" IS NULL

            OR Hour IS NULL
            OR Hour < 0
            OR Hour > 23

            OR COALESCE(
                "Rented Bike Count" < 0,
                false
            )
        ) AS quarantine_required

    FROM base
    """
)

print("quality_evaluated создано")


quality_evaluated создано


In [23]:
connection.execute(
    """
    CREATE OR REPLACE TEMP VIEW staging_data AS

    SELECT
        Date AS date,
        "Rented Bike Count" AS rented_bike_count,
        Hour AS hour,
        "Temperature(°C)" AS temperature_c,
        "Humidity(%)" AS humidity_pct,
        "Wind speed (m/s)" AS wind_speed_ms,
        "Visibility (10m)" AS visibility_10m,
        "Dew point temperature(°C)"
            AS dew_point_temperature_c,
        "Solar Radiation (MJ/m2)"
            AS solar_radiation_mj_m2,
        "Rainfall(mm)" AS rainfall_mm,
        "Snowfall (cm)" AS snowfall_cm,
        Seasons AS season,
        Holiday AS holiday,
        "Functioning Day" AS functioning_day,

        ingested_on AS source_ingested_on,

        q_valid_humidity
            AS quality_flag_invalid_humidity,

        q_functioning_day_consistency
            AS quality_flag_functioning_day_inconsistent

    FROM quality_evaluated

    WHERE NOT quarantine_required
    """
)

print("staging_data создано")


staging_data создано


In [24]:
connection.execute(
    """
    CREATE OR REPLACE TEMP VIEW quarantine_data AS

    SELECT
        Date,
        "Rented Bike Count",
        Hour,
        "Temperature(°C)",
        "Humidity(%)",
        "Wind speed (m/s)",
        "Visibility (10m)",
        "Dew point temperature(°C)",
        "Solar Radiation (MJ/m2)",
        "Rainfall(mm)",
        "Snowfall (cm)",
        Seasons,
        Holiday,
        "Functioning Day",
        ingested_on,

        concat_ws(
            ', ',
            CASE
                WHEN q_unique_date_hour
                THEN 'unique_date_hour'
            END,
            CASE
                WHEN q_rented_count_not_null
                THEN 'rented_count_not_null'
            END,
            CASE
                WHEN q_valid_hour
                THEN 'valid_hour'
            END,
            CASE
                WHEN q_non_negative_rented_count
                THEN 'non_negative_rented_count'
            END
        ) AS quality_rule_ids,

        concat_ws(
            '; ',
            CASE
                WHEN q_unique_date_hour
                THEN 'Дублирование бизнес-ключа Date + Hour'
            END,
            CASE
                WHEN q_rented_count_not_null
                THEN 'Отсутствует Rented Bike Count'
            END,
            CASE
                WHEN q_valid_hour
                THEN 'Hour находится вне диапазона 0-23'
            END,
            CASE
                WHEN q_non_negative_rented_count
                THEN 'Rented Bike Count имеет отрицательное значение'
            END
        ) AS quarantine_reason

    FROM quality_evaluated

    WHERE quarantine_required
    """
)

print("quarantine_data создано")


quarantine_data создано


In [25]:
connection.sql(
    """
    SELECT
        (SELECT COUNT(*) FROM raw_data)
            AS raw_rows,

        (SELECT COUNT(*) FROM staging_data)
            AS staging_rows,

        (SELECT COUNT(*) FROM quarantine_data)
            AS quarantine_rows,

        (
            SELECT COUNT(*) FROM staging_data
        )
        +
        (
            SELECT COUNT(*) FROM quarantine_data
        )
            AS result_rows
    """
).show()


┌──────────┬──────────────┬─────────────────┬─────────────┐
│ raw_rows │ staging_rows │ quarantine_rows │ result_rows │
│  int64   │    int64     │      int64      │    int64    │
├──────────┼──────────────┼─────────────────┼─────────────┤
│     8760 │         8760 │               0 │        8760 │
└──────────┴──────────────┴─────────────────┴─────────────┘



In [26]:
connection.sql(
    """
    DESCRIBE staging_data
    """
).show()


┌───────────────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│                column_name                │ column_type │  null   │   key   │ default │  extra  │
│                  varchar                  │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ date                                      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ rented_bike_count                         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ hour                                      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ temperature_c                             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ humidity_pct                              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ wind_speed_ms                             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │


In [27]:
connection.sql(
    """
    SELECT *
    FROM staging_data
    LIMIT 5
    """
).show()


┌────────────┬───────────────────┬───────┬───────────────┬──────────────┬───────────────┬────────────────┬─────────────────────────┬───────────────────────┬─────────────┬─────────────┬─────────┬────────────┬─────────────────┬────────────────────┬───────────────────────────────┬───────────────────────────────────────────┐
│    date    │ rented_bike_count │ hour  │ temperature_c │ humidity_pct │ wind_speed_ms │ visibility_10m │ dew_point_temperature_c │ solar_radiation_mj_m2 │ rainfall_mm │ snowfall_cm │ season  │  holiday   │ functioning_day │ source_ingested_on │ quality_flag_invalid_humidity │ quality_flag_functioning_day_inconsistent │
│    date    │       int64       │ int64 │    double     │    int64     │    double     │     int64      │         double          │        double         │   double    │   double    │ varchar │  varchar   │     boolean     │        date        │            boolean            │                  boolean                  │
├────────────┼─────────────────

## 5. Сохранение результатов в SeaweedFS

После применения правил качества нормализованные данные сохраняются в формате Parquet в слой `staging`.

Строки с критическими нарушениями сохраняются отдельно в `quarantine.parquet`.

Оба объекта привязаны к дате загрузки исходного raw-объекта через `source_ingested_on`.


In [28]:
connection.execute(
    f"""
    COPY (
        SELECT *
        FROM staging_data
    )
    TO '{STAGING_PATH}'
    (FORMAT PARQUET)
    """
)

print("staging.parquet сохранён:", STAGING_PATH)


staging.parquet сохранён: s3://staging/seoul_bike_sharing_demand/source_ingested_on=2026-10-13/staging.parquet


In [29]:
connection.execute(
    f"""
    COPY (
        SELECT *
        FROM quarantine_data
    )
    TO '{QUARANTINE_PATH}'
    (FORMAT PARQUET)
    """
)

print("quarantine.parquet сохранён:", QUARANTINE_PATH)


quarantine.parquet сохранён: s3://staging/seoul_bike_sharing_demand/source_ingested_on=2026-10-13/quarantine.parquet


In [30]:
connection.sql(
    f"""
    SELECT COUNT(*) AS staging_rows
    FROM read_parquet('{STAGING_PATH}')
    """
).show()


┌──────────────┐
│ staging_rows │
│    int64     │
├──────────────┤
│         8760 │
└──────────────┘



In [31]:
connection.sql(
    f"""
    SELECT COUNT(*) AS quarantine_rows
    FROM read_parquet('{QUARANTINE_PATH}')
    """
).show()


┌─────────────────┐
│ quarantine_rows │
│      int64      │
├─────────────────┤
│               0 │
└─────────────────┘



In [32]:
connection.sql(
    f"""
    SELECT *
    FROM read_parquet('{STAGING_PATH}')
    ORDER BY date, hour
    LIMIT 5
    """
).show()


┌────────────┬───────────────────┬───────┬───────────────┬──────────────┬───────────────┬────────────────┬─────────────────────────┬───────────────────────┬─────────────┬─────────────┬─────────┬────────────┬─────────────────┬────────────────────┬───────────────────────────────┬───────────────────────────────────────────┐
│    date    │ rented_bike_count │ hour  │ temperature_c │ humidity_pct │ wind_speed_ms │ visibility_10m │ dew_point_temperature_c │ solar_radiation_mj_m2 │ rainfall_mm │ snowfall_cm │ season  │  holiday   │ functioning_day │ source_ingested_on │ quality_flag_invalid_humidity │ quality_flag_functioning_day_inconsistent │
│    date    │       int64       │ int64 │    double     │    int64     │    double     │     int64      │         double          │        double         │   double    │   double    │ varchar │  varchar   │     boolean     │        date        │            boolean            │                  boolean                  │
├────────────┼─────────────────

## 6. Проверка качества сохранённого staging

После сохранения Parquet критические правила качества повторно применяются к уже записанному объекту `staging.parquet`.

Это позволяет убедиться, что критические нарушения действительно отсутствуют в итоговом слое.


In [33]:
connection.sql(
    f"""
    SELECT
        COUNT(*) - COUNT(DISTINCT (date, hour))
            AS duplicate_date_hour,

        COUNT(*) FILTER (
            WHERE rented_bike_count IS NULL
        ) AS rented_count_nulls,

        COUNT(*) FILTER (
            WHERE hour IS NULL
               OR hour < 0
               OR hour > 23
        ) AS invalid_hour,

        COUNT(*) FILTER (
            WHERE rented_bike_count < 0
        ) AS negative_rented_count

    FROM read_parquet('{STAGING_PATH}')
    """
).show()


┌─────────────────────┬────────────────────┬──────────────┬───────────────────────┐
│ duplicate_date_hour │ rented_count_nulls │ invalid_hour │ negative_rented_count │
│        int64        │       int64        │    int64     │         int64         │
├─────────────────────┼────────────────────┼──────────────┼───────────────────────┤
│                   0 │                  0 │            0 │                     0 │
└─────────────────────┴────────────────────┴──────────────┴───────────────────────┘



In [34]:
connection.sql(
    f"""
    SELECT
        COUNT(*) FILTER (
            WHERE quality_flag_invalid_humidity
        ) AS invalid_humidity_flags,

        COUNT(*) FILTER (
            WHERE quality_flag_functioning_day_inconsistent
        ) AS functioning_day_flags

    FROM read_parquet('{STAGING_PATH}')
    """
).show()


┌────────────────────────┬───────────────────────┐
│ invalid_humidity_flags │ functioning_day_flags │
│         int64          │         int64         │
├────────────────────────┼───────────────────────┤
│                      0 │                     0 │
└────────────────────────┴───────────────────────┘



### Вывод по формированию staging и quarantine

После применения правил качества в слой `staging` сохранено 8760 строк.

Строки, нарушающие критические правила качества, отсутствуют, поэтому `quarantine.parquet` содержит 0 записей.

Баланс данных соблюдён:

`8760 raw = 8760 staging + 0 quarantine`.

Повторная проверка сохранённого `staging.parquet` не выявила дубликатов бизнес-ключа, пропусков основной метрики, недопустимых значений часа и отрицательного количества аренд.

Предупреждающие quality-флаги также отсутствуют.


## 6. Проверка сохранённых результатов

После записи результатов в SeaweedFS файлы `staging.parquet` и `quarantine.parquet` повторно читаются через DuckDB.

Это позволяет убедиться, что результаты были корректно сохранены и доступны непосредственно из объектного хранилища.


In [35]:
connection.sql(
    f"""
    SELECT COUNT(*) AS staging_rows
    FROM read_parquet('{STAGING_PATH}')
    """
).show()


┌──────────────┐
│ staging_rows │
│    int64     │
├──────────────┤
│         8760 │
└──────────────┘



In [36]:
connection.sql(
    f"""
    SELECT COUNT(*) AS quarantine_rows
    FROM read_parquet('{QUARANTINE_PATH}')
    """
).show()


┌─────────────────┐
│ quarantine_rows │
│      int64      │
├─────────────────┤
│               0 │
└─────────────────┘



In [37]:
connection.sql(
    f"""
    SELECT *
    FROM read_parquet('{STAGING_PATH}')
    ORDER BY date, hour
    LIMIT 5
    """
).show()


┌────────────┬───────────────────┬───────┬───────────────┬──────────────┬───────────────┬────────────────┬─────────────────────────┬───────────────────────┬─────────────┬─────────────┬─────────┬────────────┬─────────────────┬────────────────────┬───────────────────────────────┬───────────────────────────────────────────┐
│    date    │ rented_bike_count │ hour  │ temperature_c │ humidity_pct │ wind_speed_ms │ visibility_10m │ dew_point_temperature_c │ solar_radiation_mj_m2 │ rainfall_mm │ snowfall_cm │ season  │  holiday   │ functioning_day │ source_ingested_on │ quality_flag_invalid_humidity │ quality_flag_functioning_day_inconsistent │
│    date    │       int64       │ int64 │    double     │    int64     │    double     │     int64      │         double          │        double         │   double    │   double    │ varchar │  varchar   │     boolean     │        date        │            boolean            │                  boolean                  │
├────────────┼─────────────────

In [38]:
connection.sql(
    f"""
    SELECT
        COUNT(*) - COUNT(DISTINCT (date, hour))
            AS duplicate_date_hour,

        COUNT(*) FILTER (
            WHERE rented_bike_count IS NULL
        ) AS rented_count_nulls,

        COUNT(*) FILTER (
            WHERE hour IS NULL
               OR hour < 0
               OR hour > 23
        ) AS invalid_hour,

        COUNT(*) FILTER (
            WHERE rented_bike_count < 0
        ) AS negative_rented_count

    FROM read_parquet('{STAGING_PATH}')
    """
).show()


┌─────────────────────┬────────────────────┬──────────────┬───────────────────────┐
│ duplicate_date_hour │ rented_count_nulls │ invalid_hour │ negative_rented_count │
│        int64        │       int64        │    int64     │         int64         │
├─────────────────────┼────────────────────┼──────────────┼───────────────────────┤
│                   0 │                  0 │            0 │                     0 │
└─────────────────────┴────────────────────┴──────────────┴───────────────────────┘



In [39]:
connection.sql(
    f"""
    SELECT
        COUNT(*) FILTER (
            WHERE quality_flag_invalid_humidity
        ) AS invalid_humidity_flags,

        COUNT(*) FILTER (
            WHERE quality_flag_functioning_day_inconsistent
        ) AS functioning_day_flags

    FROM read_parquet('{STAGING_PATH}')
    """
).show()


┌────────────────────────┬───────────────────────┐
│ invalid_humidity_flags │ functioning_day_flags │
│         int64          │         int64         │
├────────────────────────┼───────────────────────┤
│                      0 │                     0 │
└────────────────────────┴───────────────────────┘



### Вывод по проверке сохранённых данных

Файлы `staging.parquet` и `quarantine.parquet` успешно прочитаны непосредственно из SeaweedFS.

В `staging.parquet` содержится 8760 строк, в `quarantine.parquet` — 0 строк.

Повторная проверка итогового staging не выявила:
- дубликатов бизнес-ключа `date + hour`;
- пропусков `rented_bike_count`;
- недопустимых значений `hour`;
- отрицательных значений количества аренд.

Предупреждающие quality-флаги также отсутствуют.
